In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from hmmlearn.hmm import GaussianHMM

import vectorbt as vbt

In [3]:
def load_data(symbols=('SPY', '^VIX'), start='2010-01-01', auto_adjust=False):
    raw_data = yf.download(
        list(symbols),
        start=start,
        auto_adjust=auto_adjust,
        progress=False
    )

    if raw_data.empty:
        raise ValueError('No se pudieron descargar datos desde yfinance.')

    close_data = raw_data['Close'] if isinstance(raw_data.columns, pd.MultiIndex) else raw_data

    price_symbol, vix_symbol = symbols
    df = pd.DataFrame({
        'price': close_data[price_symbol],
        'vix': close_data[vix_symbol]
    }).dropna()

    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    return df

In [4]:
def compute_features(df):
    df = df.copy()
    
    # Log returns
    df['log_ret'] = np.log(df['price'] / df['price'].shift(1))
    
    # ATR (simplificado)
    df['atr'] = df['price'].rolling(14).std()
    
    # SMA diff
    sma10 = df['price'].rolling(10).mean()
    sma30 = df['price'].rolling(30).mean()
    df['sma_diff'] = sma10 - sma30
    
    # RSI (simple)
    delta = df['price'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    df = df.dropna()
    
    features = df[['log_ret', 'vix', 'atr', 'sma_diff', 'rsi']]
    
    return df, features

In [5]:
class HMMRegimeModel:
    
    def __init__(self, n_states=4):
        self.model = GaussianHMM(
            n_components=n_states,
            covariance_type="full",
            n_iter=1000,
            random_state=42
        )
        
    def fit(self, X):
        self.model.fit(X)
        
    def predict(self, X):
        return self.model.predict(X)

In [6]:
def map_states(df, states):
    df = df.copy()
    df['state'] = states
    
    summary = df.groupby('state').agg({
        'log_ret': ['mean', 'std'],
        'vix': 'mean'
    })
    
    print(summary)
    
    return df, summary

In [7]:
def trend_strategy(df):
    sma10 = df['price'].rolling(10).mean()
    sma30 = df['price'].rolling(30).mean()
    
    entries = sma10 > sma30
    exits = sma10 < sma30
    
    return entries, exits

def mean_reversion_strategy(df):
    entries = df['rsi'] < 30
    exits = df['rsi'] > 50
    
    return entries, exits

In [8]:
def apply_hmm_filter(entries, states, allowed_states):
    return entries & states.isin(allowed_states)

In [9]:
def split_data(df, train_size=0.7):
    if not 0 < train_size < 1:
        raise ValueError('train_size debe ser un float entre 0 y 1.')

    split_idx = int(len(df) * train_size)

    if split_idx <= 0 or split_idx >= len(df):
        raise ValueError('La particion generada deja una muestra vacia.')

    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    return train_df, test_df

In [10]:
REGIME_POLICY = {
    "trend": [0, 1],        # solo en bullish
    "mean_rev": [3],        # lateral
}

In [ ]:
def run_baseline(sample_df, init_cash=10_000):
    """
    Ejecuta trend y mean reversion SIN filtro HMM.
    Sirve como referencia para comparar contra la version con modelo.
    """
    sample_df, _ = compute_features(sample_df)

    trend_entries, trend_exits = trend_strategy(sample_df)
    mr_entries, mr_exits = mean_reversion_strategy(sample_df)

    trend_pf = vbt.Portfolio.from_signals(
        sample_df['price'],
        entries=trend_entries,
        exits=trend_exits,
        init_cash=init_cash,
        freq='D'
    )

    mr_pf = vbt.Portfolio.from_signals(
        sample_df['price'],
        entries=mr_entries,
        exits=mr_exits,
        init_cash=init_cash,
        freq='D'
    )

    return {'trend_pf': trend_pf, 'mr_pf': mr_pf, 'df': sample_df}

In [16]:
def run_backtest(train_df, test_df=None, n_states=4):
    train_df, train_features = compute_features(train_df)

    hmm = HMMRegimeModel(n_states=n_states)
    hmm.fit(train_features.values)

    def build_portfolios(sample_df):
        sample_df, sample_features = compute_features(sample_df)
        states = pd.Series(hmm.predict(sample_features.values), index=sample_features.index)

        sample_df = sample_df.loc[sample_features.index].copy()
        sample_df['state'] = states

        trend_entries, trend_exits = trend_strategy(sample_df)
        mr_entries, mr_exits = mean_reversion_strategy(sample_df)

        trend_entries_f = apply_hmm_filter(
            trend_entries,
            sample_df['state'],
            REGIME_POLICY['trend']
        )
        mr_entries_f = apply_hmm_filter(
            mr_entries,
            sample_df['state'],
            REGIME_POLICY['mean_rev']
        )

        trend_pf = vbt.Portfolio.from_signals(
            sample_df['price'],
            entries=trend_entries_f,
            exits=trend_exits,
            init_cash=10_000,
            freq='D'
        )

        mr_pf = vbt.Portfolio.from_signals(
            sample_df['price'],
            entries=mr_entries_f,
            exits=mr_exits,
            init_cash=10_000,
            freq='D'
        )

        return {
            'trend_pf': trend_pf,
            'mr_pf': mr_pf,
            'df': sample_df
        }

    results = {
        'in_sample': build_portfolios(train_df),
        'model': hmm
    }

    if test_df is not None:
        results['out_of_sample'] = build_portfolios(test_df)

    return results

In [ ]:
DEFAULT_METRICS = [
    'Total Return [%]',
    'Sharpe Ratio',
    'Sortino Ratio',
    'Max Drawdown [%]',
    'Win Rate [%]',
    'Total Trades',
    'Profit Factor',
    'Expectancy',
]

def compare_stats(portfolios, metrics=None):
    """
    Genera una tabla comparativa de métricas para múltiples portfolios.
    Función reutilizable para cualquier conjunto de estrategias vbt.

    Parámetros
    ----------
    portfolios : dict {label: vbt.Portfolio}
    metrics    : list de nombres de métricas de vbt.stats().
                 Si None, usa DEFAULT_METRICS.

    Retorna
    -------
    pd.DataFrame con métricas como filas y estrategias como columnas.
    """
    if metrics is None:
        metrics = DEFAULT_METRICS

    records = {}
    for label, pf in portfolios.items():
        s = pf.stats()
        records[label] = {m: (s[m] if m in s.index else None) for m in metrics}

    return pd.DataFrame(records, index=metrics).round(3)

In [ ]:
STATE_COLORS = ['#aed6f1', '#a9dfbf', '#f9e79f', '#f1948a']

def plot_regimes(df, title='Regímenes HMM sobre el Precio', figsize=(14, 5)):
    """
    Visualiza el precio con fondo coloreado por régimen HMM.

    Parámetros
    ----------
    df      : DataFrame con columnas 'price' y 'state' y DatetimeIndex.
    title   : Título del gráfico.
    figsize : Tamaño de la figura.
    """
    unique_states = sorted(df['state'].unique())
    colors = {s: STATE_COLORS[i % len(STATE_COLORS)] for i, s in enumerate(unique_states)}

    if 'log_ret' in df.columns and 'vix' in df.columns:
        summary = df.groupby('state').agg({'log_ret': 'mean', 'vix': 'mean'})
        labels = {
            s: f'Estado {s}  |  ret={summary.loc[s, "log_ret"]:.4f}  |  vix={summary.loc[s, "vix"]:.1f}'
            for s in unique_states
        }
    else:
        labels = {s: f'Estado {s}' for s in unique_states}

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(df.index, df['price'], color='black', linewidth=1, zorder=2)

    states = df['state'].values
    dates  = df.index
    prev   = states[0]
    start  = dates[0]

    for i in range(1, len(states)):
        if states[i] != prev:
            ax.axvspan(start, dates[i], alpha=0.35, color=colors[prev], linewidth=0)
            prev  = states[i]
            start = dates[i]
    ax.axvspan(start, dates[-1], alpha=0.35, color=colors[prev], linewidth=0)

    patches = [
        mpatches.Patch(color=colors[s], alpha=0.7, label=labels[s])
        for s in unique_states
    ]
    price_line = plt.Line2D([0], [0], color='black', linewidth=1, label='Precio')
    ax.legend(handles=[price_line] + patches, loc='upper left', fontsize=8)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Precio')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Carga y partición ─────────────────────────────────────────────────────────
df = load_data()
train_df, test_df = split_data(df, train_size=0.7)

# ── Baseline (sin HMM) ────────────────────────────────────────────────────────
baseline_is  = run_baseline(train_df)
baseline_oos = run_baseline(test_df)

# ── HMM Backtest ──────────────────────────────────────────────────────────────
results = run_backtest(train_df=train_df, test_df=test_df)

# ── Tabla comparativa ─────────────────────────────────────────────────────────
portfolios = {
    'Trend  Baseline IS':  baseline_is['trend_pf'],
    'Trend  HMM IS':       results['in_sample']['trend_pf'],
    'Trend  Baseline OOS': baseline_oos['trend_pf'],
    'Trend  HMM OOS':      results['out_of_sample']['trend_pf'],
    'MR  Baseline IS':     baseline_is['mr_pf'],
    'MR  HMM IS':          results['in_sample']['mr_pf'],
    'MR  Baseline OOS':    baseline_oos['mr_pf'],
    'MR  HMM OOS':         results['out_of_sample']['mr_pf'],
}

stats_table = compare_stats(portfolios)
display(stats_table)

# ── Visualización de regímenes ────────────────────────────────────────────────
plot_regimes(results['in_sample']['df'],     title='Regímenes HMM — In-Sample')
plot_regimes(results['out_of_sample']['df'], title='Regímenes HMM — Out-of-Sample')

In-Sample Trend Strategy:
Start                         2010-03-29 00:00:00
End                           2021-05-07 00:00:00
Period                         2798 days 00:00:00
Start Value                               10000.0
End Value                            13289.387402
Total Return [%]                        32.893874
Benchmark Return [%]                   259.802247
Max Gross Exposure [%]                      100.0
Total Fees Paid                               0.0
Max Drawdown [%]                        16.620001
Max Drawdown Duration           825 days 00:00:00
Total Trades                                   33
Total Closed Trades                            33
Total Open Trades                               0
Open Trade PnL                                0.0
Win Rate [%]                            45.454545
Best Trade [%]                          14.351363
Worst Trade [%]                         -5.855534
Avg Winning Trade [%]                    4.674018
Avg Losing Trade [%]    